# 004 R1–R4 — Colab T4 候选排序手动入口（骨架，未执行）

第一条模型路线 R1–R4 的手动运行入口。**本 notebook 是骨架：只定义入口与产物口径，不含实测分数。**
在 Colab T4 上逐格手动运行；不等同训练完成或提分。

- 不硬编码本地路径与当前 test IDs：候选池、manifest、权重目录一律从环境变量/上传路径读入，分子集合从输入数据动态读取。
- Q/L/C 固定同一份：single 按不依赖真值的规则选一张 Q；融合只能读 Q；先同极性/同 adduct 融合，再比较跨条件聚合。
- 产物写入 `artifacts/004-research/<run-id>/`：config、environment、metrics、checkpoint、日志、逐分子表（列见 004 计划）。
- 参考：`docs/plan/004-t4-candidate-ranking.md` 第一条模型路线与“需要实现的代码”节；模型定义见 `src/models/fingerprint.py`（许可 unknown）；位映射见 `src/chemistry.py`。


In [ ]:
# C0：公共配置（无硬编码路径/ID）。路径由环境变量或上传位置传入。
import json, os, sys
from pathlib import Path

RUN_ID = os.environ.get("RUN_ID", "004-r1-manual")  # 手动设置，如 004-r1-20261001
MANIFEST_PATH = Path(os.environ.get("MANIFEST_PATH", "/content/np_manifest.json"))
CANDIDATE_DIR = Path(os.environ.get("CANDIDATE_DIR", "/content/candidates"))
WEIGHTS_DIR = Path(os.environ.get("WEIGHTS_DIR", "/content/weights"))
FP_BITS_PATH = Path(os.environ.get("FP_BITS_PATH", "/content/fp_bits.npy"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", f"/content/004-output/{RUN_ID}"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = int(os.environ.get("SEED", "2026"))
MAX_PEAKS, NBITS = 128, 6930
print({"run_id": RUN_ID, "manifest": str(MANIFEST_PATH), "candidates": str(CANDIDATE_DIR), "weights": str(WEIGHTS_DIR), "output": str(OUTPUT_DIR), "seed": SEED})


## R1：固定 Q/L/C — mass-only → cosine → FPNet single

问题：真值在哪一步丢失？神经指纹是否提供排序增量？同一 Q/L/C 下比较三步的真值召回与 MRR@25/Top1/Top25（候选存在条件下的排序另计）。质量窗口 5/10/20ppm 逐步观察。


In [ ]:
# R1 手动入口（需在 T4 上运行；本地不执行）。
# 1) 从 MANIFEST_PATH 读固定 Q/L（行号与 key 动态读取，不硬编码 ID）。
# 2) 从 CANDIDATE_DIR 读结构候选库 C（packed 指纹，按 batch 搬；禁止全池 unpack）。
# 3) 同一 Q/L/C 依次跑 mass-only、cosine（src/retrieval）、FPNet single（src/models/fingerprint + src/ranking.score_single_spectrum）。
# 4) 逐分子记录 true_in_pool / true_after_mass / true_after_truncation / rank_before / rank_after / rr。
from pathlib import Path
print("R1 placeholder: run mass-only -> cosine -> FPNet single on fixed Q/L/C; save per-molecule table to", str(OUTPUT_DIR))


## R2：同一候选池 — single / logit 均值 / 候选级融合 / merged 对照

问题：多谱收益来自哪里？跨条件混合是否反而损失信息？先同极性/同 adduct 融合，再比较跨条件聚合；merged 模型只喂合并峰表（见 fingerprint.py 路由注释）。


In [ ]:
# R2 手动入口（需在 T4 上运行）。
# - single：不依赖真值的规则选一张 Q（规则写入 config）。
# - logit 均值：src/ranking.aggregate_mean_scores（候选级均值，默认规则）。
# - 列表级对照：src/ranking.round_robin_fuse（真正轮转；旧 retrieval.aggregate 非轮转已修正）。
# - merged 对照：fp_merged 权重 + 合并峰表（_merge_peaks 口径），不得把单谱输入记成 merged 验证。
# 全部全局 <=25 截断并计数 n_truncated。
print("R2 placeholder: compare single / mean / round-robin / merged on same C; record n_truncated")


## R3：目标域训练 — 冻结头 / 末两层解冻 / 从头对照

问题：目标域适配是否有效？公开 checkpoint 暴露未知与严格泛化分开解释。先 200–500 步小预算估计（batch 16/32，峰 128），按实测 step 时间再定完整轮数。T4 用 FP16 AMP；BCE + 同质量窗口排序损失；保存 optimizer/config/split。


In [ ]:
# R3 手动入口（需在 T4 上运行；不对随机假输入训练）。
# from src.train_fingerprint import estimate_budget, TrainConfig, train_step_torch
# 1) estimate_budget() 列出 200/500 步 x batch 16/32 配置，逐个手动跑并记录 seconds/step 与峰值显存。
# 2) 三模式：head_only -> unfreeze_last2 -> from_scratch（训练侧排除留出 key，manifest 路径写入 config）。
# 3) AMP FP16 + GradScaler；m/z/正弦嵌入保持所需精度，不照搬 TPU BF16。
# 4) save_train_state() 保存 model+optimizer+config+split；推理 ckpt 无确认优化器状态，只作微调初始化。
print("R3 placeholder: estimate 200-500 steps x batch 16/32, then three training modes; save optimizer/config/split")


## R4：简单重排器 — FPNet + cosine/analog/质量/元数据

问题：同候选集下能否修正互补错误？OOF 或独立 query 组生成 ranker 样本，不从验证标签调负例。


In [ ]:
# R4 手动入口。
# from src.ranking import SimpleLinearRanker, RANKER_FEATURES
# 特征列固定 [fp_dot, spectral_cosine, neg_abs_ppm, adduct_match, instrument_match]；
# fit_mean_difference 仅作占位试探，样本必须来自 OOF/独立 query 组；未拟合 predict 抛错。
print("R4 placeholder: fit SimpleLinearRanker on OOF/independent groups; evaluate on same C")


## 产物与记录

每个实验把 config、environment、metrics、checkpoint、日志、逐分子表写入 `OUTPUT_DIR`（对应 `artifacts/004-research/<run-id>/`），再人工摘录到 `docs/progress/004-t4-candidate-ranking.md`。只报告实测指标；小观察集同时给命中数量与逐分子变化。


In [ ]:
# 产物骨架：调用方按实际结果填充后落盘（示例键，不写分数）。
import json
result = {"run_id": RUN_ID, "seed": SEED, "status": "not run (skeleton)", "metrics": {}, "per_molecule_table": []}
(OUTPUT_DIR / "result_skeleton.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("wrote", str(OUTPUT_DIR / "result_skeleton.json"))
